**EE344 - Final Project**

Authors: Eeshani, Kuhu, Ruslana

**Question being answered:** Using audio features and popularity metrics from the 2023 Spotify streaming dataset, can we predict whether a song will be nominated for a Grammy in major song-related categories?
(Categories Considered: Song of the Year, Best R&B Song, Best Rap Song, Best Country Song, Best Song Written for Visual Media)

This file preprocesses the data and test out a few baseline models (Decision Tree, Random Forest, Random Forest with grid searched hyper parameters). Next steps include iterating on these models to solve the severe class imbalance problem this data set presents

# Data Preprocessing



In [115]:
import pandas as pd
import numpy as np
import re
import unicodedata
import matplotlib.pyplot as plt
import seaborn as sns

#imports for the preprocessing and modeling
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc
from sklearn.model_selection import GridSearchCV

In [116]:
# list of all nominees for 2024 best song in Song of the Year, Best R&B Song, Best Rap Song, Best Country Song, Best Song Written for Visual Media
nominee_list = [
  "Billie Eilish - What Was I Made For?",
  "Olivia Rodrigo - Vampire",
  "SZA - Kill Bill",
  "Miley Cyrus - Flowers",
  "Dua Lipa - Dance The Night",
  "Jon Batiste - Butterfly",
  "Taylor Swift - Anti-Hero",
  "Lana Del Rey - A&W",

  "SZA - Snooze",
  "Victoria Monét - On My Mama",
  "Coco Jones - ICU",
  "Robert Glasper - Back To Love",
  "Halle - Angel",

  "Killer Mike - SCIENTISTS & ENGINEERS",
  "Drake - Rich Flex",
  "Lil Uzi Vert - Just Wanna Rock",
  "Nicki Minaj - Barbie World",
  "Doja Cat - Attention",

  "Chris Stapleton - White Horse",
  "Morgan Wallen - Last Night",
  "Tyler Childers - In Your Love",
  "Zach Bryan - I Remember Everything",
  "Brandy Clark - Buried",

  "Rihanna - Lift Me Up",
  "Ryan Gosling - I'm Just Ken",

  "Billie Eilish - What Was I Made For? [From The Motion Picture \"Barbie\"]",
  "Dua Lipa - Dance The Night (From Barbie The Album)",
  "Nicki Minaj - Barbie World (with Aqua) [From Barbie The Album]",
  "Rihanna - Lift Me Up - From Black Panther: Wakanda Forever - Music From and Inspired By"
]

In [117]:
DATA_PATH = "/content/spotify-2023.csv"
df = pd.read_csv(DATA_PATH, encoding="latin1")

print(df.shape)
df.head()

(953, 24)


,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6


In [118]:
#add in a nominee column into the dataset before preprocessing anything
def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

#normalize nominee list
nominee_set = set(normalize(x) for x in nominee_list)

#combining artist and song into same format
df["song_artist"] = df["artist(s)_name"] + " - " + df["track_name"]

# normalize it
df["song_artist_norm"] = df["song_artist"].apply(normalize)

df["nominee"] = df["song_artist_norm"].isin(nominee_set).astype(int)

df[["artist(s)_name","track_name","nominee"]].head(20)

,artist(s)_name,track_name,nominee
0,"Latto, Jung Kook",Seven (feat. Latto) (Explicit Ver.),0
1,Myke Towers,LALA,0
2,Olivia Rodrigo,vampire,1
3,Taylor Swift,Cruel Summer,0
4,Bad Bunny,WHERE SHE GOES,0
5,"Dave, Central Cee",Sprinter,0
6,"Eslabon Armado, Peso Pluma",Ella Baila Sola,0
7,Quevedo,Columbia,0
8,Gunna,fukumean,0
9,"Peso Pluma, Yng Lvcas",La Bebe - Remix,0


In [119]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   track_name            953 non-null    object
 1   artist(s)_name        953 non-null    object
 2   artist_count          953 non-null    int64 
 3   released_year         953 non-null    int64 
 4   released_month        953 non-null    int64 
 5   released_day          953 non-null    int64 
 6   in_spotify_playlists  953 non-null    int64 
 7   in_spotify_charts     953 non-null    int64 
 8   streams               953 non-null    object
 9   in_apple_playlists    953 non-null    int64 
 10  in_apple_charts       953 non-null    int64 
 11  in_deezer_playlists   953 non-null    object
 12  in_deezer_charts      953 non-null    int64 
 13  in_shazam_charts      903 non-null    object
 14  bpm                   953 non-null    int64 
 15  key                   858 non-null    ob

Observations:


*   Streams, in_deezer_playlists, and in_shazaam_charts has object data type when it should be integers.
*   object data type (except streams) should be the categorical features (mode, and key)



In [120]:
#converting object types to ints
df['streams'] = pd.to_numeric(df['streams'], errors= 'coerce')
df['in_deezer_playlists'] = pd.to_numeric(df['in_deezer_playlists'], errors= 'coerce')
df['in_shazam_charts'] = pd.to_numeric(df['in_shazam_charts'], errors= 'coerce')

#sum of all the nulls in each col
df.isnull().sum()


,0
track_name,0
artist(s)_name,0
artist_count,0
released_year,0
released_month,0
released_day,0
in_spotify_playlists,0
in_spotify_charts,0
streams,1
in_apple_playlists,0


In [121]:
#fill in the null values to 0 if integer, NA if object
df['in_shazam_charts'] = df['in_shazam_charts'].fillna(0)
df['in_deezer_playlists'] = df['in_deezer_playlists'].fillna(0)
df['streams'] = df['streams'].fillna(0)
df['key']= df['key'].fillna('NA')

df.isnull().sum() #is 0 nulls now

,0
track_name,0
artist(s)_name,0
artist_count,0
released_year,0
released_month,0
released_day,0
in_spotify_playlists,0
in_spotify_charts,0
streams,0
in_apple_playlists,0


Observations:

Below we tried to remove the songs outside of the grammy nomination window for 2024. The performance of our models decreased when we did this as noted by the F1 score. The removal process took away ~600 rows leaving only 286 data points to train on which is not enough for ML models to train on.

Next decision to iterate on: keep all songs, get rid of dates for time series relevance, see how that impacts the evaluation metrics.

In [122]:
# #remove the dates if not released between the grammy window October 1, 2022, through September 15, 2023
# df["release_date"] = pd.to_datetime(
#     df[["released_year", "released_month", "released_day"]]
#     .rename(columns={
#         "released_year": "year",
#         "released_month": "month",
#         "released_day": "day"
#     })
# )
# # df[["released_year","released_month","released_day","release_date"]].head()

# start_date = pd.Timestamp("2022-10-01")
# end_date = pd.Timestamp("2023-09-15")

# before = len(df)
# df = df[(df["release_date"] >= start_date) & (df["release_date"] <= end_date)]
# after = len(df)
# print("Rows before:", before)
# print("Rows after:", after)
# print("Removed:", before - after)

# print(df["release_date"].min())
# print(df["release_date"].max())

In [123]:
#drop dates because this is not timeseries data
df = df.drop(columns=[
    "released_year",
    "released_month",
    "released_day"])

#verify
df.head()

,track_name,artist(s)_name,artist_count,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,in_apple_charts,in_deezer_playlists,in_deezer_charts,...,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%,song_artist,song_artist_norm,nominee
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,553,147,141381703.0,43,263,45.0,10,...,80,89,83,31,0,8,4,"Latto, Jung Kook - Seven (feat. Latto) (Explic...",latto jung kook seven feat latto explicit ver,0
1,LALA,Myke Towers,1,1474,48,133716286.0,48,126,58.0,14,...,71,61,74,7,0,10,4,Myke Towers - LALA,myke towers lala,0
2,vampire,Olivia Rodrigo,1,1397,113,140003974.0,94,207,91.0,14,...,51,32,53,17,0,31,6,Olivia Rodrigo - vampire,olivia rodrigo vampire,1
3,Cruel Summer,Taylor Swift,1,7858,100,800840817.0,116,207,125.0,12,...,55,58,72,11,0,11,15,Taylor Swift - Cruel Summer,taylor swift cruel summer,0
4,WHERE SHE GOES,Bad Bunny,1,3133,50,303236322.0,84,133,87.0,15,...,65,23,80,14,63,11,6,Bad Bunny - WHERE SHE GOES,bad bunny where she goes,0


In [124]:
#one hot encode key and mode
cat_cols = ["key", "mode"]
id_cols = ["track_name", "artist(s)_name", "song_artist", "song_artist_norm"]
id_data = df[id_cols]
target = "nominee"

X_raw = df.drop(columns=id_cols + [target]).copy()
y = df[target].copy()

num_cols = [c for c in X_raw.columns if c not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", "passthrough", num_cols)
    ],
    remainder="drop"
)

X_encoded = preprocessor.fit_transform(X_raw)
ohe = preprocessor.named_transformers_["cat"]
ohe_cols = ohe.get_feature_names_out(cat_cols)

final_cols = list(ohe_cols) + list(num_cols)

X = pd.DataFrame(X_encoded, columns=final_cols)
X.head()

,key_A,key_A#,key_B,key_C#,key_D,key_D#,key_E,key_F,key_F#,key_G,...,in_deezer_charts,in_shazam_charts,bpm,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10.0,826.0,125.0,80.0,89.0,83.0,31.0,0.0,8.0,4.0
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.0,382.0,92.0,71.0,61.0,74.0,7.0,0.0,10.0,4.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,14.0,949.0,138.0,51.0,32.0,53.0,17.0,0.0,31.0,6.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,12.0,548.0,170.0,55.0,58.0,72.0,11.0,0.0,11.0,15.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,15.0,425.0,144.0,65.0,23.0,80.0,14.0,63.0,11.0,6.0


In [125]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   key_A                 953 non-null    float64
 1   key_A#                953 non-null    float64
 2   key_B                 953 non-null    float64
 3   key_C#                953 non-null    float64
 4   key_D                 953 non-null    float64
 5   key_D#                953 non-null    float64
 6   key_E                 953 non-null    float64
 7   key_F                 953 non-null    float64
 8   key_F#                953 non-null    float64
 9   key_G                 953 non-null    float64
 10  key_G#                953 non-null    float64
 11  key_NA                953 non-null    float64
 12  mode_Major            953 non-null    float64
 13  mode_Minor            953 non-null    float64
 14  artist_count          953 non-null    float64
 15  in_spotify_playlists  9

In [127]:
# ruslana's part

#baseline models: decision tree, random forest with grid searched parameters

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

#baseline model 1 decision tree
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)
print("Classification report on Decision Tree: \n")
print(classification_report(y_test, y_pred))

#baseline model 2 decision tree
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print("Classification report on Random Forest: \n")
print(classification_report(y_test, y_pred))

# grid search to find best params on random forest classifier
param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_
print("Best parameters for random forest:", grid.best_params_)

# evaluate best model
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print(cm)

y_prob = best_model.predict_proba(X_test)[:,1]
roc = roc_auc_score(y_test, y_prob)
precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall, precision)

print("ROC AUC (grid searched random forest):", roc)
print("PR AUC (grid searched random forest):", pr_auc)

# feature importance
importances = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importances.head(10)

Classification report on Decision Tree: 

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       283
           1       0.00      0.00      0.00         3

    accuracy                           0.98       286
   macro avg       0.49      0.50      0.50       286
weighted avg       0.98      0.98      0.98       286

Classification report on Random Forest: 

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       283
           1       0.00      0.00      0.00         3

    accuracy                           0.99       286
   macro avg       0.49      0.50      0.50       286
weighted avg       0.98      0.99      0.98       286



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Best parameters for random forest: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       283
           1       0.00      0.00      0.00         3

    accuracy                           0.99       286
   macro avg       0.49      0.50      0.50       286
weighted avg       0.98      0.99      0.98       286

[[283   0]
 [  3   0]]
ROC AUC (grid searched random forest): 0.7290930506478209
PR AUC (grid searched random forest): 0.0439595807242866


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,0
in_spotify_charts,0.121056
bpm,0.113540
in_deezer_charts,0.091530
acousticness_%,0.084190
danceability_%,0.066233
in_apple_charts,0.061792
energy_%,0.058582
valence_%,0.055491
in_apple_playlists,0.053153
in_spotify_playlists,0.047770
